## Notebook 3 - Silver Layer Notebook

### Load Bronze Tables

In [ ]:
from pyspark.sql.functions import (
    col, trim, lower, upper, when,
    to_date, year, row_number
)
from pyspark.sql.window import Window

electricity = spark.table("carbonops.bronze.electricity_consumption")
fuel = spark.table("carbonops.bronze.fuel_consumption")
facilities = spark.table("carbonops.bronze.facilities")
production = spark.table("carbonops.bronze.production")
emission_factors = spark.table("carbonops.bronze.emission_factors")
carbon_targets = spark.table("carbonops.bronze.carbon_targets")

print("Bronze tables loaded successfully!")

### Clean Electricity Consumption Data

In [ ]:
electricity_clean = (
    electricity
    .withColumn("period_end", to_date(col("period_end")))
    .withColumn("received_at", to_date(col("received_at")))
    .withColumn("electricity_value", col("electricity_value").cast("double"))
    .withColumn("electricity_unit", lower(trim(col("electricity_unit"))))
    .filter(col("facility_id").isNotNull())
    .filter(col("period_end").isNotNull())
    .filter(col("electricity_value") >= 0)
    .withColumn(
        "electricity_kwh",
        when(col("electricity_unit") == "kwh", col("electricity_value"))
        .when(col("electricity_unit") == "mwh", col("electricity_value") * 1000)
    )
    .filter(col("electricity_kwh").isNotNull())
)

# Keep latest record for each record_id
window_spec = Window.partitionBy("record_id").orderBy(
    col("received_at").desc_nulls_last()
)

electricity_clean = (
    electricity_clean
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

display(electricity_clean)

### Clean Fuel Consumption Data

In [ ]:
fuel_clean = (
    fuel
    .withColumn("period_end", to_date(col("period_end")))
    .withColumn("received_at", to_date(col("received_at")))
    .withColumn("fuel_type", lower(trim(col("fuel_type"))))
    .withColumn("unit", lower(trim(col("unit"))))
    .withColumn("quantity", col("quantity").cast("double"))
    .filter(col("facility_id").isNotNull())
    .filter(col("period_end").isNotNull())
    .filter(col("quantity") >= 0)
)

# Standardize common unit spellings
fuel_clean = (
    fuel_clean
    .withColumn(
        "unit",
        when(col("unit").isin("litres", "liters", "l"), "litre")
        .when(col("unit").isin("m³", "cubic_meter", "cubic_metre"), "m3")
        .when(col("unit").isin("kilograms", "kilogram"), "kg")
        .otherwise(col("unit"))
    )
)

# Deduplicate by record_id
fuel_window = Window.partitionBy("record_id").orderBy(
    col("received_at").desc_nulls_last()
)

fuel_clean = (
    fuel_clean
    .withColumn("row_num", row_number().over(fuel_window))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

display(fuel_clean)

### Clean Facility Master Data

In [ ]:
facilities_clean = (
    facilities
    .withColumn("facility_id", trim(col("facility_id")))
    .withColumn("facility_name", trim(col("facility_name")))
    .withColumn("city", trim(col("city")))
    .withColumn("country", trim(col("country")))
    .withColumn("status", trim(col("status")))
    .dropDuplicates(["facility_id"])
    .filter(col("facility_id").isNotNull())
)

display(facilities_clean)

### Enrich Electricity Data with Facility Information

In [ ]:
electricity_enriched = (
    electricity_clean
    .join(
        facilities_clean.select(
            "facility_id",
            "facility_name",
            "city",
            "state",
            "country",
            "industry"
        ),
        on="facility_id",
        how="left"
    )
)

display(electricity_enriched)

### Enrich Fuel Data with Facility Information

In [ ]:
fuel_enriched = (
    fuel_clean
    .join(
        facilities_clean.select(
            "facility_id",
            "facility_name",
            "city",
            "state",
            "country",
            "industry"
        ),
        on="facility_id",
        how="left"
    )
)

display(fuel_enriched)

### Clean Production Data

In [ ]:
production_clean = (
    production
    .withColumn("period_end", to_date(col("period_end")))
    .withColumn("production_units", col("production_units").cast("double"))
    .withColumn("operating_hours", col("operating_hours").cast("double"))
    .filter(col("facility_id").isNotNull())
    .filter(col("period_end").isNotNull())
    .filter(col("production_units") >= 0)
    .dropDuplicates(["record_id"])
)

display(production_clean)

### Clean Emission Factors

In [ ]:
emission_factors_clean = (
    emission_factors
    .withColumn("source_type", lower(trim(col("source_type"))))
    .withColumn("activity_unit", lower(trim(col("activity_unit"))))
    .withColumn("year", col("year").cast("int"))
    .withColumn(
        "kg_co2e_per_unit",
        col("kg_co2e_per_unit").cast("double")
    )
    .filter(col("kg_co2e_per_unit") >= 0)
    .dropDuplicates(["factor_id"])
)

display(emission_factors_clean)

### Clean Carbon Targets

In [ ]:
carbon_targets_clean = (
    carbon_targets
    .withColumn("year", col("year").cast("int"))
    .withColumn(
        "annual_target_tco2e",
        col("annual_target_tco2e").cast("double")
    )
    .filter(col("facility_id").isNotNull())
    .filter(col("annual_target_tco2e") >= 0)
    .dropDuplicates(["facility_id", "year"])
)

display(carbon_targets_clean)

### Write Silver Delta Tables

In [ ]:
silver_tables = {
    "electricity_clean": electricity_clean,
    "electricity_enriched": electricity_enriched,
    "fuel_clean": fuel_clean,
    "fuel_enriched": fuel_enriched,
    "facilities_clean": facilities_clean,
    "production_clean": production_clean,
    "emission_factors_clean": emission_factors_clean,
    "carbon_targets_clean": carbon_targets_clean
}

for table_name, df in silver_tables.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"carbonops.silver.{table_name}")
    )

print("All CarbonOps Silver tables created successfully!")

### Validate Silver Tables

In [ ]:
for table_name in silver_tables:
    count = spark.table(f"carbonops.silver.{table_name}").count()
    print(f"{table_name}: {count} records")

display(spark.table("carbonops.silver.electricity_enriched"))
display(spark.table("carbonops.silver.fuel_enriched"))